# Create a simple forcing to run the tests of SIM
The same code can be used to create higher resolution forcings

In [24]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import cmocean.cm as cmo
import datetime
import glob

In [2]:
forcing_dir = "/storage/jrieck/SIM_forcing/"
path = "/aos/home/jrieck/src/SIM/"

This is the same code that is used to create the bathymetry and mask in [create_SIM_grid_mask](create_SIM_grid_mask.ipynb).

In [3]:
def create_long_lat(dx, nx, ny, beta_rot, grid="T"):
    r_earth = 6370.0 # radius of the Earth
    beta    = beta_rot   # to rotate the mask with respect to Greenwich
                     # when beta = 0, x and x1 axis are aligned with Greenwich
    if grid=="T":
        if int(80 % dx)==0:
            dx_pole = 2480 + (80 - (dx / 2))  # distance of pole from tracer point of cell 0,0
            dy_pole = 2240 + (80 - (dx / 2))
        elif int(32 % dx)==0:
            dx_pole = 2464 + (32 - (dx / 2))  # distance of pole from tracer point of cell 0,0
            dy_pole = 2208 + (32 - (dx / 2))
    elif grid=="U":
        if int(80 % dx)==0:
            dx_pole = 2480 + (80 - dx)  # distance of pole from U point of cell 0,0
            dy_pole = 2240 + (80 - (dx / 2))
        elif int(32 % dx)==0:
            dx_pole = 2464 + (32 - dx)  # distance of pole from U point of cell 0,0
            dy_pole = 2208 + (32 - (dx / 2))
        nx -= 1 # U grid is one smaller in x
    elif grid=="V":
        if int(80 % dx)==0:
            dx_pole = 2480 + (80 - (dx / 2))  # distance of pole from V point of cell 0,0
            dy_pole = 2240 + (80 - dx)
        elif int(32 % dx)==0:
            dx_pole = 2464 + (32 - (dx / 2))  # distance of pole from V point of cell 0,0
            dy_pole = 2208 + (32 - dx)
        ny -= 1 # V grid is one smaller in y
    elif grid == "B":
        if int(80 % dx)==0:
            dx_pole = 2480 + (80 - dx)  # distance of pole from V point of cell 0,0
            dy_pole = 2240 + (80 - dx)
        elif int(32 % dx)==0:
            dx_pole = 2464 + (32 - dx)
            dy_pole = 2208 + (32 - dx)
        nx -= 1 # to make the dimensions fit the others
        ny -= 1
    deg2rad = np.pi / 180
    rad2deg = 180 / np.pi
          
    latmin  = 90.0
    ilatmin = 0
    jlatmin = 0
    r1max   = 0
    lat = np.zeros((nx, ny))
    long = np.zeros((nx, ny))

    ii = np.arange(0, nx)
    jj = np.arange(0, ny)
    j, i = np.meshgrid(jj, ii)
    #---------- position on the stereographic plane -------------------------------
    x1 = i * dx - dx_pole
    y1 = j * dx - dy_pole
    r1 = np.sqrt(x1**2 + y1**2)
    if np.max(r1) > r1max:
        r1max = np.max(r1)
    #---------- angle of the cone -------------------------------------------------
    tanteta  = r1 / (2 * r_earth)
    #----------- short radius on the sphere ---------------------------------------
    r = 2 * r_earth * tanteta / (1 + tanteta**2)
    #----------- get the latitude and longitude -----------------------------------
    lat = np.arccos(r / r_earth) * rad2deg
    if np.min(lat < latmin):
        latmin = np.min(lat)
        ilatmin = i[np.unravel_index(lat.argmin(), lat.shape)]
        jlatmin = j[np.unravel_index(lat.argmin(), lat.shape)]
    x1 = np.where(((x1 >= 0.0) & (x1 < 1e-8)), 1e-8, x1)
    long = np.where(((x1 >= 0.0) & (y1 >= 0)), np.arctan(y1 / x1) * rad2deg + beta, long)
    long = np.where(((x1 >= 0.0) & (y1 < 0)), np.arctan(y1 / x1) * rad2deg + beta + 360.0, long)
    long = np.where((x1 < 0.0), np.arctan(y1 / x1) * rad2deg + beta + 180.0, long)
    long = np.where((long >= 360.0), long - 360.0, long)
    return long, lat

Define functions used later to rotate the u- and v-vectors onto the grid.

In [4]:
# used later for rotation of wind
def local_longitude(lon, lat):
    deg2rad = np.pi / 180
    lon_r = lon.T * deg2rad
    lat_r = lat.T * deg2rad
    # centered difference along the i-axis (grid x-direction)
    dlon = np.roll(lon_r, -1, axis=0) - np.roll(lon_r, 1, axis=0)
    dlon = (dlon + np.pi) % (2 * np.pi) - np.pi   # wrap to [-pi, pi]
    dlat = np.roll(lat_r, -1, axis=0) - np.roll(lat_r, 1, axis=0)
    L = np.arctan2(dlat, dlon * np.cos(lat_r))
    # one-sided differences at the i-boundaries
    L[0, :]  = L[1, :]
    L[-1, :] = L[-2, :]
    return L

In [5]:
def rotate_u(u, v, L):
    return u * np.cos(L) + v * np.sin(L)

In [6]:
def rotate_v(u, v, L):
    return -u * np.sin(L) + v * np.cos(L)

## Create grid and mask
Create the 32 km-grid

In [7]:
dx_new = 32.0
nx_new = 160
ny_new = 135
beta_rot = 32.0

In [8]:
lonT_new, latT_new = create_long_lat(dx_new, nx_new+2, ny_new+2, 32.0, grid="T")

Load 32 km mask

In [9]:
mask_tmp = np.loadtxt(path + "/src/mask_dx32.00_nx160_ny135.dat", dtype=str)
mask32 = np.zeros((ny_new+2, nx_new+2))
for j in range(0, ny_new+2):
    for i in range(0, nx_new+2):
        mask32[j, i] = int(mask_tmp[j][i])

## Atmosphere and ocean temperature

Load atmospheric and oceanic temperature forcing data. The data comes from [ERA5 monthly means](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-monthly-means?tab=overview) and we add two points in x-direction on either side for the interpolation.

In [10]:
airT_era = xr.open_dataset("/storage/jrieck/ERA5/northOf55/monthly/ERA5_2000-2025_2m_temperature.grib").roll(longitude=720, roll_coords=True)
ocnT_era = xr.open_dataset("/storage/jrieck/ERA5/northOf55/monthly/ERA5_2000-2025_sea_surface_temperature.grib").roll(longitude=720, roll_coords=True)
ocnT_era["sst"] = ocnT_era.sst.fillna(-1e12) - 273.15
periodic = xr.Dataset()
periodic["time"] = (("time"), xr.date_range(start=str(airT_era.time[0].values), end=str(airT_era.time[-1].values), freq="1MS"))
periodic["longitude"] = (("longitude"), np.hstack([-0.5, -0.25, airT_era.longitude.where(airT_era.longitude >= 0, airT_era.longitude + 360), 360., 360.25]))
periodic["latitude"] = (("latitude"), airT_era.latitude.data)
periodic["t2m"] = (("time", "latitude", "longitude"), np.concatenate([airT_era.t2m.isel(longitude=slice(-2, -1)).values,
                                                                      airT_era.t2m.isel(longitude=slice(-1, None)).values,
                                                                      airT_era.t2m.values, 
                                                                      airT_era.t2m.isel(longitude=slice(0, 1)).values,
                                                                      airT_era.t2m.isel(longitude=slice(1, 2)).values], axis=2))
periodic["sst"] = (("time", "latitude", "longitude"), np.concatenate([ocnT_era.sst.isel(longitude=slice(-2, -1)).values,
                                                                      ocnT_era.sst.isel(longitude=slice(-1, None)).values,
                                                                      ocnT_era.sst.values, 
                                                                      ocnT_era.sst.isel(longitude=slice(0, 1)).values,
                                                                      ocnT_era.sst.isel(longitude=slice(1, 2)).values], axis=2))

Create a `xarray` dataset with the new grid.

In [11]:
gridT_new = xr.Dataset()
gridT_new["lon"] = (("y", "x"), lonT_new.T)
gridT_new["lat"] = (("y", "x"), latT_new.T)
gridT_new["mask"] = (("y", "x"), mask32)

Interpolate the atmospheric temperatures onto the SIM-grid and add a time dimension that the model understands.

In [12]:
airT_out = periodic.t2m.interp(longitude=gridT_new.lon, latitude=gridT_new.lat, method="pchip")

In [13]:
airT_out["time"] = (("time"), [int((periodic.time.values[i].astype('datetime64[s]').item() 
                                - datetime.datetime(1900,1,1)).total_seconds() / 3600) for i in range(0, len(periodic.time))])
airT_out["time"].attrs = {"units": "hours since 1900-1-1"}

Write to disk. We only write 2006-12 - 2008-02 to be able to run some tests and to save space in the git repository.

In [14]:
airT_out.rename("air").sel(time=slice("2006-12-01", "2008-02-28")).to_netcdf(forcing_dir + "airT/air.mon.mean_dx" + str(dx_new) + "0_nx" + str(nx_new) + "_ny" + str(ny_new) + ".nc")

Oceanic temperatures are only needed as monthly climatolgy, so we cimpute that, then interpolate to the SIM-grid and fill in the NaNs etc.

In [15]:
ocnT_clim = periodic.sst.groupby("time.month").mean()
ocnT_tmp = ocnT_clim.interp(longitude=gridT_new.lon, latitude=gridT_new.lat, method="pchip")
ocnT_tmp = ocnT_tmp.where(((gridT_new.mask == 1) & (ocnT_tmp>-100)), other=np.nan)
ocnT_tmp = ocnT_tmp.interpolate_na("y", method="nearest", fill_value="extrapolate")
ocnT_out = ocnT_tmp.where(gridT_new.mask==1, other=-4)

Ocean temperatures need to be stored in a text file similar to the mask etc.

In [16]:
for m in range(0, 12):
    l = " " + "\n ".join(["\t".join(row.astype(str)) for row in ocnT_out.isel(month=m).values])
    with open(forcing_dir + "ocnT/Tocn" + f"{m+1:02d}" + "_dx" + str(dx_new) + "0_nx" + str(nx_new) + "_ny" + str(ny_new), "w") as n:
        n.write(l)

## Ocean currents

Climatological ocean currents are interpolated from old forcing. Those might be replaced with ocean reanalysis data at some point (*e.g.* GLORYS)

In [17]:
dx_old = 20
nx_old = 258
ny_old = 218

Loading ocean currents on the old 20-km grid and create `xarray` datasets.

In [18]:
uwater_tmp = np.loadtxt(forcing_dir + "ocncurrent/20/uwater20.clim", dtype=None)
vwater_tmp = np.loadtxt(forcing_dir + "ocncurrent/20/vwater20.clim", dtype=None)

In [19]:
uwater = xr.Dataset()
uwater["x"] = (("x"), -(2480 + (80 - dx_old)) + np.arange(nx_old+1) * dx_old)
uwater["y"] = (("y"), -(2240 + (80 - (dx_old / 2))) + np.arange(ny_old) * dx_old)
uwater["uwater"] = (("y", "x"), uwater_tmp)

vwater = xr.Dataset()
vwater["x"] = (("x"), -(2480 + (80 - (dx_old / 2))) + np.arange(nx_old) * dx_old)
vwater["y"] = (("y"), -(2240 + (80 - dx_old)) + np.arange(ny_old+1) * dx_old)
vwater["vwater"] = (("y", "x"), vwater_tmp)

Interpolate the 20-km currents to the new 32-km grid.

In [20]:
uwater_out = uwater.uwater.interp(x=-(2464 + (32 - dx_new)) + np.arange(nx_new+1) * dx_new, 
                                  y=-(2208 + (32 - (dx_new / 2))) + np.arange(ny_new) * dx_new ,
                                  method="pchip")
vwater_out = vwater.vwater.interp(x=-(2464 + (32 - (dx_new / 2))) + np.arange(nx_new) * dx_new, 
                                  y=-(2208 + (32 - dx_new)) + np.arange(ny_new+1) * dx_new ,
                                  method="pchip")

Fill in NaNs

In [21]:
uwater_out = uwater_out.interpolate_na("y", method="nearest", fill_value="extrapolate").interpolate_na("x", method="nearest", fill_value="extrapolate")
vwater_out = vwater_out.interpolate_na("y", method="nearest", fill_value="extrapolate").interpolate_na("x", method="nearest", fill_value="extrapolate")

Write ocean currents to text files.

In [22]:
l = " " + "\n ".join(["\t".join(row.astype(str)) for row in uwater_out.values])
with open(forcing_dir + "ocncurrent/uwater_dx" + str(dx_new) + "0_nx" + str(nx_new) + "_ny" + str(ny_new) + ".clim", "w") as n:
    n.write(l)

l = " " + "\n ".join(["\t".join(row.astype(str)) for row in vwater_out.values])
with open(forcing_dir + "ocncurrent/vwater_dx" + str(dx_new) + "0_nx" + str(nx_new) + "_ny" + str(ny_new) + ".clim", "w") as n:
    n.write(l)

## Wind

Wind forcing is computed from hourly [ERA5](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=overview) data, subsampled to 6-hourly.  
The wind forcing takes up most space on disk due to its high temporal resolution. To be able to include the files to run the tests in the git repository, we create two files, each containing just a one of data to run the tests.  
As for ERA5 temperatures, we convert longitude to `range(0, 360)` and add some halo points in the x-direction for the interpolation.

In [27]:
udata = xr.open_mfdataset(glob.glob("/storage/jrieck/ERA5/northOf55/ERA5_200[6-8]??_10m_u_component_of_wind.grib")).isel(time=slice(0, None, 6))
vdata = xr.open_mfdataset(glob.glob("/storage/jrieck/ERA5/northOf55/ERA5_200[6-8]??_10m_v_component_of_wind.grib")).isel(time=slice(0, None, 6))
udata["longitude"] = udata.longitude.where(udata.longitude >= 0, udata.longitude + 360)
udata = udata.sortby("longitude")
vdata["longitude"] = vdata.longitude.where(vdata.longitude >= 0, vdata.longitude + 360)
vdata = vdata.sortby("longitude")

# extend data by 2 in x direction for smooth interpolation across -180/180 boundary
# u and v from ERA5 are on the same longitude and latitude, se we use only the ones from udata
periodic = xr.Dataset()
periodic["time"] = (("time"), udata.valid_time.data)
periodic["longitude"] = (("longitude"), np.hstack([-0.5, -0.25, udata.longitude.where(udata.longitude >= 0, udata.longitude + 360), 360., 360.25]))
periodic["latitude"] = (("latitude"), udata.latitude.data)
periodic["u10"] = (("time", "latitude", "longitude"), np.concatenate([udata.u10.isel(longitude=slice(-2, -1)).values,
                                                                      udata.u10.isel(longitude=slice(-1, None)).values,
                                                                      udata.u10.values, 
                                                                      udata.u10.isel(longitude=slice(0, 1)).values,
                                                                      udata.u10.isel(longitude=slice(1, 2)).values], axis=2))
periodic["v10"] = (("time", "latitude", "longitude"), np.concatenate([vdata.v10.isel(longitude=slice(-2, -1)).values,
                                                                      vdata.v10.isel(longitude=slice(-1, None)).values,
                                                                      vdata.v10.values, 
                                                                      vdata.v10.isel(longitude=slice(0, 1)).values,
                                                                      vdata.v10.isel(longitude=slice(1, 2)).values], axis=2))

Create the B-grid for the wind forcing

In [28]:
lonB_new, latB_new = create_long_lat(dx_new, nx_new+2, ny_new+2, 32.0, grid="B")

In [29]:
gridB_new = xr.Dataset()
gridB_new["lon"] = (("y", "x"), lonB_new.T)
gridB_new["lat"] = (("y", "x"), latB_new.T)

Interpolate `u` and `v` onto the B-grid of the model. Note that in ERA5, `u` and `v` are the zonal and meridional velocities, respectively. We, however, need the velocities in the `x`- and `y`-directions of the model's grid. Thus we need to rotate u and v onto our grid.

In [31]:
u_out7 = periodic.u10.sel(time=slice("2006-12-31 12:00:00", "2007-01-02 12:00:00")).interp(longitude=gridB_new.lon, latitude=gridB_new.lat, method="pchip")
v_out7 = periodic.v10.sel(time=slice("2006-12-31 12:00:00", "2007-01-02 12:00:00")).interp(longitude=gridB_new.lon, latitude=gridB_new.lat, method="pchip")

In [32]:
u_out8 = periodic.u10.sel(time=slice("2007-12-31 12:00:00", "2008-01-02 12:00:00")).interp(longitude=gridB_new.lon, latitude=gridB_new.lat, method="pchip")
v_out8 = periodic.v10.sel(time=slice("2007-12-31 12:00:00", "2008-01-02 12:00:00")).interp(longitude=gridB_new.lon, latitude=gridB_new.lat, method="pchip")

In [33]:
L = local_longitude(lonB_new, latB_new)
u_rot7 = rotate_u(u_out7, v_out7, L)
v_rot7 = rotate_v(u_out7, v_out7, L)
u_rot8 = rotate_u(u_out8, v_out8, L)
v_rot8 = rotate_v(u_out8, v_out8, L)

Create a dataset with the wind forcing.

In [34]:
out7 = xr.Dataset()
out7["time"] = u_out7.time
out7["lon"] = (("y", "x"), u_out7.longitude.data)
out7["lat"] = (("y", "x"), u_out7.latitude.data)
out7["uwnd"] = (("time", "y", "x"), u_rot7.data)
out7["vwnd"] = (("time", "y", "x"), v_rot7.data)

In [35]:
out8 = xr.Dataset()
out8["time"] = u_out8.time
out8["lon"] = (("y", "x"), u_out8.longitude.data)
out8["lat"] = (("y", "x"), u_out8.latitude.data)
out8["uwnd"] = (("time", "y", "x"), u_rot8.data)
out8["vwnd"] = (("time", "y", "x"), v_rot8.data)

Add a time dimension that the model understands and write the forcing to a file. As for the 

In [36]:
out7["time"] = (("time"), [int((out7.time.values[i].astype('datetime64[s]').item() 
                                - datetime.datetime(1900,1,1)).total_seconds() / 3600) for i in range(0, len(out7.time))])
out7["time"].attrs = {"units": "hours since 1900-1-1"}

In [37]:
out8["time"] = (("time"), [int((out8.time.values[i].astype('datetime64[s]').item() 
                                - datetime.datetime(1900,1,1)).total_seconds() / 3600) for i in range(0, len(out8.time))])
out8["time"].attrs = {"units": "hours since 1900-1-1"}

In [40]:
out7.to_netcdf(forcing_dir + "wind/wind_dx" + str(dx_new) + "0_nx" + str(nx_new) + "_ny" + str(ny_new) + "_2007.nc")

In [41]:
out8.to_netcdf(forcing_dir + "wind/wind_dx" + str(dx_new) + "0_nx" + str(nx_new) + "_ny" + str(ny_new) + "_2008.nc")